# MiniGPT: a decoder-only character-level language model

A GPT-style model trained from scratch on ~2.8M characters of Dostoevsky, Kafka, Plato, Shakespeare and Blake.

All layers, including attention, are hand-written in [`transformer.py`](transformer.py). No `nn.Transformer`, no `scaled_dot_product_attention`.

Sections in this notebook:
1. Data and tokenizer: characters, or the from-scratch BPE in [`bpe.py`](bpe.py)
2. Model, training (AdamW, warmup + cosine LR, gradient clipping)
3. Sampling with temperature and top-k
4. **KV cache**: why generation doesn't recompute the whole sequence every token
5. **Attention heatmaps** from the trained model
6. **Attention variants**: causal / bidirectional / sliding-window / padding masks, and MHA vs GQA vs MQA

The encoder-decoder half (cross-attention) is in [`seq2seq.ipynb`](seq2seq.ipynb).

In [ ]:
import os, time, math
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from bpe import BPE
from transformer import MiniGPT, Attention

torch.manual_seed(1337)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SMOKE = os.environ.get("SMOKE") == "1"   # tiny run to check the notebook executes end to end
TOKENIZER = os.environ.get("TOKENIZER", "char")   # "char" or "bpe"
print("Using device:", device)

## Data

In [ ]:
files = [
    "data/TheIdiot.txt",
    "data/theMetamorphosis.txt",
    "data/TheRepublic.txt",
    "data/shakespeareSonnets.txt",
    "data/williamBlakePoems.txt",
]
text = "\n".join(open(f, encoding="utf-8").read() for f in files)
print("Total characters:", len(text))

## Tokenizer

- **char**: each distinct character is one token (vocab of about 119). Simple, but a 256-token context only covers 256 characters.
- **bpe**: byte-level BPE with 1024 tokens (see `bpe.py`). Common words become single tokens, so the same context covers about 2.6x more text.

Loss per token isn't comparable between the two, so training also reports **bits per character**.

There's no padding token: the model handles any length up to `context_length`, so prompts are fed in as they are.

In [ ]:
if TOKENIZER == "bpe":
    tok = BPE.train(text, vocab_size=1024)               # ~40s
    encode, decode, vocab_size = tok.encode, tok.decode, tok.vocab_size
else:
    chars = sorted(set(text))
    vocab_size = len(chars)
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for ch, i in stoi.items()}

    def encode(s):
        unknown = set(s) - stoi.keys()
        assert not unknown, f"characters not in vocab: {unknown}"
        return [stoi[c] for c in s]

    def decode(ids):
        return "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long, device=device)
chars_per_token = len(text) / len(data)
print(f"tokenizer: {TOKENIZER} | {chars_per_token:.2f} chars/token")
split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]   # the model never trains on val_data
print("Vocab size:", vocab_size, "| train tokens:", len(train_data), "| val tokens:", len(val_data))

## Model

Each block is `x + SelfAttention(LN(x))`, then `x + MLP(LN(x))`: pre-LayerNorm with residual connections, as in GPT-2. Self-attention uses `causal=True`, so position *t* only sees positions ≤ *t*.

In [ ]:
context_length = 256
dim, n_blocks, n_heads = 128, 6, 4

model = MiniGPT(vocab_size, context_length, dim, n_blocks, n_heads, dropout=0.1).to(device)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")

## Training

In [ ]:
batch_size = 64
max_steps = 30 if SMOKE else 5000
eval_every = 10 if SMOKE else 500
warmup, peak_lr = 200, 1e-3

def get_batch(split):
    src = train_data if split == "train" else val_data
    ix = torch.randint(len(src) - context_length - 1, (batch_size,), device=device)
    x = torch.stack([src[i : i + context_length] for i in ix])
    y = torch.stack([src[i + 1 : i + context_length + 1] for i in ix])   # next char = target
    return x, y

def loss_fn(logits, y):
    return F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))

@torch.no_grad()
def estimate_loss(iters=2 if SMOKE else 50):
    model.eval()
    out = {s: sum(loss_fn(model(x), y).item() for x, y in (get_batch(s) for _ in range(iters))) / iters
           for s in ("train", "val")}
    model.train()
    return out

def lr_at(step):   # linear warmup, then cosine decay to 10% of peak
    if step < warmup:
        return (step + 1) / warmup
    progress = (step - warmup) / max(1, max_steps - warmup)
    return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress))

optimizer = torch.optim.AdamW(model.parameters(), lr=peak_lr, weight_decay=0.1)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_at)

history = []
model.train()
for step in range(max_steps + 1):
    if step % eval_every == 0:
        losses = estimate_loss()
        history.append((step, losses["train"], losses["val"]))
        bpc = losses["val"] / math.log(2) / chars_per_token
        print(f"step {step:5d} | train {losses['train']:.4f} | val {losses['val']:.4f} | val {bpc:.3f} bits/char")
    if step == max_steps:
        break
    x, y = get_batch("train")
    loss = loss_fn(model(x), y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()

torch.save(model.state_dict(), "minigpt.pt")

In [ ]:
steps, tr, va = zip(*history)
plt.plot(steps, tr, label="train")
plt.plot(steps, va, label="val")
plt.xlabel("step"); plt.ylabel("cross-entropy (nats/token)"); plt.legend(); plt.title("Training curve")
plt.show()

## Text generation

The model predicts a distribution over the next character, we sample one, append it and repeat. **Temperature** sharpens (<1) or flattens (>1) the distribution. **Top-k** samples only from the k most likely characters.

In [ ]:
prompt = torch.tensor([encode("The rain was pouring")], device=device)
print(decode(model.generate(prompt, 500, temperature=0.8, top_k=20)[0].tolist()))

## KV cache

Without a cache, generating token *t* re-runs attention over all *t* previous tokens, and that work is thrown away and redone on the next step.

With a cache, each layer keeps the **keys and values** it has already computed. A new token only computes its own Q, K and V and attends over the cached K/V. Per-token cost drops from a full forward pass over the context to a single-position pass.

`test_transformer.py` checks that both paths produce the same logits.

In [ ]:
def time_generation(use_cache, n=50 if SMOKE else 300):
    torch.manual_seed(0)
    start = time.perf_counter()
    model.generate(prompt, n, use_cache=use_cache)
    if device.type == "cuda":
        torch.cuda.synchronize()
    return time.perf_counter() - start

no_cache, with_cache = time_generation(False), time_generation(True)
print(f"without cache: {no_cache:.2f}s | with cache: {with_cache:.2f}s | {no_cache / with_cache:.1f}x faster")

## What the attention heads look at

Row *i* is the query token and the columns show how much it attends to each earlier token. The upper triangle is empty because of the causal mask.

In [ ]:
sample = "Gregor Samsa woke from troubled dreams."
model.eval()
with torch.no_grad():
    model(torch.tensor([encode(sample)], device=device))

labels = [decode([i]) for i in encode(sample)]
fig, axes = plt.subplots(2, n_heads, figsize=(4 * n_heads, 8))
for row, layer in enumerate([0, n_blocks - 1]):
    weights = model.blocks[layer].self_attn.attn_weights[0].cpu()   # (H, T, T)
    for h in range(n_heads):
        ax = axes[row, h]
        ax.imshow(weights[h], cmap="viridis")
        ax.set_title(f"layer {layer}, head {h}")
        ax.set_xticks(range(len(labels)), labels, fontsize=6)
        ax.set_yticks(range(len(labels)), labels, fontsize=6)
plt.tight_layout(); plt.show()

## Attention variants

All of these are the same `Attention` class. Only the mask, or where K/V come from, changes. White = may attend, black = blocked.

In [ ]:
T = 12
pad = torch.tensor([[False] * 8 + [True] * 4])       # last 4 positions are padding
masks = {
    "Encoder (bidirectional)": torch.ones(T, T, dtype=torch.bool),
    "Decoder (causal)":        Attention(32, 4, causal=True).build_mask(T, T, None, "cpu"),
    "Sliding window (w=4)":    Attention(32, 4, causal=True, window=4).build_mask(T, T, None, "cpu"),
    "Encoder + padding mask":  Attention(32, 4).build_mask(T, T, pad, "cpu")[0, 0].expand(T, T),
}
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, m) in zip(axes, masks.items()):
    ax.imshow(m, cmap="gray", vmin=0, vmax=1)
    ax.set_title(name); ax.set_xlabel("key"); ax.set_ylabel("query")
plt.tight_layout(); plt.show()

**MHA vs GQA vs MQA.** Multi-head attention gives every query head its own K/V head. Grouped-query attention shares one K/V head across a group of query heads, and multi-query attention shares a single K/V head across all of them. Quality stays close to MHA, while the KV cache, which is what limits long-context inference, shrinks by `n_heads / n_kv_heads`.

In [ ]:
head_dim = dim // n_heads
for name, n_kv in [("MHA", n_heads), ("GQA", 2), ("MQA", 1)]:
    variant = MiniGPT(vocab_size, context_length, dim, n_blocks, n_heads, n_kv_heads=n_kv)
    cache_bytes = n_blocks * 2 * n_kv * head_dim * context_length * 4   # K and V, fp32, full context
    print(f"{name}: kv_heads={n_kv} | params {sum(p.numel() for p in variant.parameters()) / 1e6:.2f}M "
          f"| KV cache per sequence {cache_bytes / 1024:.0f} KiB")